# 10 · Robustness  (Part 9)

Does the main conclusion survive? The headline claims:

* **A** — the efficient repricing threshold sits in a **≈4–7% band** (knee ≈ 6%).
* **B** — a cumulative-inflation threshold rule holds the real price far tighter
  than what hotels actually did, at **~half the price changes** of monthly
  indexing, out-of-sample.
* **C** — mechanical **monthly CPI indexing still loses ~15% of real value**
  during an inflation acceleration (it only adds last month's print).
* **D** — occupancy is **inelastic / not causally identified**.
* **E** — high inflation makes each price change **larger and more upward**, not
  materially more frequent.

Variations: GBA vs national CPI · include/exclude 2022 · exclude extreme
inflation (> p90) · room vs bed occupancy · β scenarios · expanding-window
selection · pass-through lag length · category set.

**Output** `outputs/tables/robustness_matrix.csv`, `robustness_regime.csv`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

SRC = Path.cwd() / "src"
sys.path.insert(0, str(SRC))
import config as C
from common import AuditLog
import policies as PL
import pricing_eval as EV

LOG = AuditLog("10_robustness")
df0 = pd.read_parquet(C.P_ANALYSIS).rename(columns={"infl_mom_gba_frac": "infl_mom",
                                                    "infl_mom_nac_frac": "infl_mom_nac_f"})
CORE = C.CORE_CATEGORIES
CATS = CORE + [C.COMPOSITE_TOTAL]
SELECT = ("2022-01-01", "2023-12-01")
HAC = dict(cov_type="HAC", cov_kwds={"maxlags": 6})


def frontier_knee(cats, select, test, defl="cpi_gba", inflc="infl_mom",
                  occ_col="room_occupancy", beta=-0.5, drop_extreme=False):
    per_tau = {t: [] for t in C.THRESHOLD_GRID}
    p1m, p1sm, obsm, p2rep, obsrep = [], [], [], {t: [] for t in C.THRESHOLD_GRID}, []
    for cat in cats:
        raw = df0[df0.hotel_category == cat].sort_values("date")
        s = raw.drop(columns=["infl_mom", "cpi_gba", "room_occupancy"]).copy()
        s["infl_mom"] = raw[inflc].to_numpy()
        s["cpi_gba"] = raw[defl].to_numpy()
        s["room_occupancy"] = raw[occ_col].to_numpy()
        if drop_extreme:
            hi = s["infl_mom"].quantile(0.90)
            s = s[s["infl_mom"] <= hi]
        occ = PL.seasonal_occ_norm(s[s.split == "train"])
        tgt = EV.target_real_rate(s)
        for t in C.THRESHOLD_GRID:
            sc = EV.score(PL.simulate(PL.policy2_threshold(t), s, select, occ_norm=occ,
                                      anchor_real=tgt), beta, tgt)
            per_tau[t].append((sc["n_reprice"], sc["mean_abs_real_dev_pct"]))
        # OOS on test
        occ_t = occ
        p1 = EV.score(PL.simulate(PL.policy1_monthly_cpi(), s, test, occ_norm=occ_t,
                                  anchor_real=tgt), beta, tgt)
        p1s = EV.score(PL.simulate(PL.policy1_monthly_cpi(), s, select, occ_norm=occ,
                                   anchor_real=tgt), beta, tgt)
        p1sm.append(p1s["mean_abs_real_dev_pct"])
        ob = EV.score(PL.observed_path(s, test), beta, tgt)
        p1m.append(p1["mean_abs_real_dev_pct"]); obsm.append(ob["mean_abs_real_dev_pct"])
        obsrep.append(ob["n_reprice"])
        for t in C.THRESHOLD_GRID:
            sc = EV.score(PL.simulate(PL.policy2_threshold(t), s, test, occ_norm=occ_t,
                                      anchor_real=tgt), beta, tgt)
            p2rep[t].append((sc["n_reprice"], sc["mean_abs_real_dev_pct"], sc["revpar_vs_obs_pct"]))
    xs = np.array([np.mean([v[0] for v in per_tau[t]]) for t in C.THRESHOLD_GRID])
    ys = np.array([np.mean([v[1] for v in per_tau[t]]) for t in C.THRESHOLD_GRID])
    xn = (xs - xs.min()) / (np.ptp(xs) + 1e-9)
    yn = (ys - ys.min()) / (np.ptp(ys) + 1e-9)
    knee = C.THRESHOLD_GRID[int(np.argmin(np.hypot(xn, yn)))]
    p2 = np.array([np.mean([v[0] for v in p2rep[knee]]),
                   np.mean([v[1] for v in p2rep[knee]]),
                   np.mean([v[2] for v in p2rep[knee]])])
    return dict(knee_tau_pct=knee * 100, p2_reprice=p2[0], p2_madev_pct=p2[1],
                p2_revpar_vs_obs_pct=p2[2], p1_madev_test_pct=np.mean(p1m),
                p1_madev_accel_pct=np.mean(p1sm),
                obs_madev_pct=np.mean(obsm), obs_reprice=np.mean(obsrep))

## Robustness matrix — claims A / B / C

In [2]:
variants = {
    "base (GBA CPI, room occ, 2022 incl.)": dict(),
    "national CPI deflator": dict(defl="cpi_nac", inflc="infl_mom_nac_f"),
    "exclude extreme inflation (>p90)": dict(drop_extreme=True),
    "bed occupancy target": dict(occ_col="bed_occupancy"),
    "β = 0 (fully inelastic)": dict(beta=0.0),
    "β = −1.0": dict(beta=-1.0),
    "core star tiers only (no composite)": dict(cats=CORE),
    "selection = 2023 only": dict(select=("2023-01-01", "2023-12-01")),
}
rows = []
for name, kw in variants.items():
    cats = kw.pop("cats", CATS)
    sel = kw.pop("select", SELECT)
    r = frontier_knee(cats, sel, C.TEST_SEGMENT, **kw)
    r["variant"] = name
    r["A_knee_in_4_7"] = 4 <= r["knee_tau_pct"] <= 7
    r["B_p2_beats_obs"] = (r["p2_madev_pct"] < r["obs_madev_pct"]) and (r["p2_reprice"] < r["obs_reprice"])
    r["C_p1_lags_in_accel"] = r["p1_madev_accel_pct"] >= 8
    rows.append(r)
rob = pd.DataFrame(rows).set_index("variant").round(2)
rob = rob[["knee_tau_pct", "A_knee_in_4_7", "p2_reprice", "p2_madev_pct", "obs_reprice",
           "obs_madev_pct", "B_p2_beats_obs", "p1_madev_accel_pct", "p1_madev_test_pct",
           "C_p1_lags_in_accel", "p2_revpar_vs_obs_pct"]]
rob.to_csv(C.OUT_TAB / "robustness_matrix.csv")
LOG.log("write", "robustness_matrix.csv", len(rob))
LOG.log("robustness_A", "knee τ across variants", detail=list(rob.knee_tau_pct))
LOG.log("robustness_B", "P2 beats observed", detail=f"{int(rob.B_p2_beats_obs.sum())}/{len(rob)} variants")
LOG.log("robustness_C", "P1 lags >=8% madev in acceleration", detail=f"{int(rob.C_p1_lags_in_accel.sum())}/{len(rob)} variants")
rob

  [10_robustness] write                  robustness_matrix.csv                   8  
  [10_robustness] robustness_A           knee τ across variants                     [6.0, 7.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0]
  [10_robustness] robustness_B           P2 beats observed                          8/8 variants
  [10_robustness] robustness_C           P1 lags >=8% madev in acceleration           7/8 variants


,knee_tau_pct,A_knee_in_4_7,p2_reprice,p2_madev_pct,obs_reprice,obs_madev_pct,B_p2_beats_obs,p1_madev_accel_pct,p1_madev_test_pct,C_p1_lags_in_accel,p2_revpar_vs_obs_pct
variant,,,,,,,,,,,
"base (GBA CPI, room occ, 2022 incl.)",6.0,True,13.8,5.88,25.2,11.21,True,16.59,4.51,True,0.00
national CPI deflator,7.0,False,11.8,6.31,25.2,11.07,True,16.67,4.44,True,-0.19
exclude extreme inflation (>p90),6.0,True,10.0,7.80,21.2,10.51,True,18.41,70.04,True,0.16
bed occupancy target,6.0,True,13.8,5.88,25.2,11.21,True,16.59,4.51,True,-0.00
β = 0 (fully inelastic),6.0,True,13.8,5.88,25.2,11.21,True,16.59,4.51,True,0.52
β = −1.0,6.0,True,13.8,5.88,25.2,11.21,True,16.59,4.51,True,0.00
core star tiers only (no composite),6.0,True,14.0,5.84,25.0,11.52,True,7.60,4.40,False,0.28
selection = 2023 only,6.0,True,13.8,5.88,25.2,11.21,True,8.91,4.51,True,0.00


## Claim E — regime behaviour under alternative CPI / occupancy / lag length

In [3]:
def regime_tests(inflc="infl_mom_gba", occ="room_occupancy"):
    d = df0[(df0.in_sample == 1) & df0.hotel_category.isin(CORE)].copy()
    d["dln_cpi"] = np.log1p(d[inflc] / 100)
    d["dln_occ"] = d.groupby("hotel_category")[occ].transform(lambda s: np.log(s).diff())
    edges = d[inflc].quantile(C.REGIME_QUANTILES).values
    d["regime"] = pd.cut(d[inflc], edges, include_lowest=True, labels=C.REGIME_LABELS)
    d = d.dropna(subset=["dln_average_rate", "dln_cpi", "regime"])
    d = d.assign(absdln=d.dln_average_rate.abs() * 100,
                 up=(d.dln_average_rate > 0).astype(float),
                 big=(d.dln_average_rate.abs() > .01).astype(float),
                 high=(d.regime == "high").astype(float),
                 catf=d.hotel_category.astype("category"))
    out = {}
    for nm, f in [("magnitude", "absdln ~ high + catf"),
                  ("asymmetry", "up ~ high + catf"),
                  ("frequency", "big ~ high + catf")]:
        m = smf.ols(f, data=d).fit(**HAC)
        out[nm] = (round(m.params["high"], 3), round(m.pvalues["high"], 4))
    return out


er = []
for nm, kw in [("GBA CPI / room occ", dict()),
               ("national CPI", dict(inflc="infl_mom_nac")),
               ("bed occupancy", dict(occ="bed_occupancy"))]:
    o = regime_tests(**kw)
    er.append(dict(variant=nm, magnitude_coef=o["magnitude"][0], magnitude_p=o["magnitude"][1],
                   asymmetry_coef=o["asymmetry"][0], asymmetry_p=o["asymmetry"][1],
                   frequency_coef=o["frequency"][0], frequency_p=o["frequency"][1]))
robE = pd.DataFrame(er).set_index("variant")
robE.to_csv(C.OUT_TAB / "robustness_regime.csv")
LOG.log("write", "robustness_regime.csv", len(robE))
robE

  [10_robustness] write                  robustness_regime.csv                   3  


,magnitude_coef,magnitude_p,asymmetry_coef,asymmetry_p,frequency_coef,frequency_p
variant,,,,,,
GBA CPI / room occ,4.393,0.0,0.194,0.0,0.088,0.0020
national CPI,4.365,0.0,0.213,0.0,0.089,0.0019
bed occupancy,4.393,0.0,0.194,0.0,0.088,0.0020


## Summary

In [4]:
surv = dict(
    A=f"knee τ* ∈ {{{int(rob.knee_tau_pct.min())}..{int(rob.knee_tau_pct.max())}}}% across "
      f"{len(rob)} variants; in the 4–7% band for {int(rob.A_knee_in_4_7.sum())}/{len(rob)}.",
    B=f"threshold rule beats observed pricing on real-price stability AND repricing "
      f"count in {int(rob.B_p2_beats_obs.sum())}/{len(rob)} variants.",
    C=f"in an inflation ACCELERATION (2022–23) monthly CPI indexing madev is "
      f"{rob.p1_madev_accel_pct.median():.0f}% (≥8% in {int(rob.C_p1_lags_in_accel.sum())}/{len(rob)}); "
      f"in the 2024–26 disinflation it is fine ({rob.p1_madev_test_pct.median():.1f}%).",
    E=f"magnitude (+{robE.magnitude_coef.mean():.1f}pp) & upward-asymmetry "
      f"(+{robE.asymmetry_coef.mean():.2f}) effects significant in "
      f"{(robE.magnitude_p < .05).sum()}/{len(robE)} & {(robE.asymmetry_p < .05).sum()}/{len(robE)} "
      f"variants; the frequency effect is also significant but an order of magnitude "
      f"smaller (+{robE.frequency_coef.mean()*100:.0f}pp from an ~90% base).",
    D="occupancy elasticity remains associational — see notebook 06 (relative-price "
      "panel ≈ 0; 2SLS weak / over-ID rejected).",
)
for k, v in surv.items():
    print(f"[{k}] {v}")
    LOG.log("survives", k, detail=v)
LOG.flush()
print("\n10 complete.")

[A] knee τ* ∈ {6..7}% across 8 variants; in the 4–7% band for 7/8.
  [10_robustness] survives               A                                          knee τ* ∈ {6..7}% across 8 variants; in the 4–7% band for 7/8.
[B] threshold rule beats observed pricing on real-price stability AND repricing count in 8/8 variants.
  [10_robustness] survives               B                                          threshold rule beats observed pricing on real-price stability AND repricing count in 8/8 variants.
[C] in an inflation ACCELERATION (2022–23) monthly CPI indexing madev is 17% (≥8% in 7/8); in the 2024–26 disinflation it is fine (4.5%).
  [10_robustness] survives               C                                          in an inflation ACCELERATION (2022–23) monthly CPI indexing madev is 17% (≥8% in 7/8); in the 2024–26 disinflation it is fine (4.5%).
[E] magnitude (+4.4pp) & upward-asymmetry (+0.20) effects significant in 3/3 & 3/3 variants; the frequency effect is also significant but an ord